# 05 · Sequence models: GRU, attention and a selective SSM (Mamba-style)

**Goal:** build the three black-box backbones from their core operations, and connect each one
to SP ideas: a recursive filter, a data-dependent weighting, and a bank of input-controlled IIR
filters.

**Time:** ~1.5 h · **You'll write:** `tokens`, `attention`, `selective_scan`

**How the exercises work.** Each one is a function with `raise NotImplementedError`. Replace that
with your code, then run the cell below it. `use(...)` checks your function against an independent
reference (the project's own code, autograd, or a closed form):
- `[ok]` → your version is used from then on.
- `[todo]` / `[fail]` → the notebook falls back to the reference solution so you can keep going.

Hints are folded under each exercise. Solutions are in `tutorials/solutions.py`; try first.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "tutorials":
    os.chdir("..")
sys.path.insert(0, "tutorials")

import numpy as np
import torch
import matplotlib.pyplot as plt
from tutor import use, close
import solutions as sol

torch.set_num_threads(4)
plt.rcParams.update({"figure.figsize": (10, 3.2), "axes.grid": True, "grid.alpha": 0.3})

import json, time
import torch.nn.functional as F
from biotwin.data.dataset import Cohort, standard_sets, subset, LH, H
from biotwin.models import neural
from biotwin.train import fit, predict
cohort = Cohort.load("data/cohort.npz")
S = standard_sets(cohort, seed=0)
te = S["test_eval"]

## 1. One token per time step

All three models read the same sequence of 84 tokens (72 past + 12 future), each with 4 numbers:

$$x_t = [\ (\text{cgm}_t - 140)/50 \text{ (0 in the future)},\quad \text{observed flag (1 past, 0 future)},\quad \text{ins}_t - \text{basal},\quad \text{carbs}_t / 10\ ]$$

The future CGM slot is zero **and** the flag says so. Without the flag, the model couldn't tell
"glucose = 140" from "unknown".

## Exercise 1 — `tokens(w, LH)` → `[B, LH+H, 4]`


<details><summary><b>Hint</b></summary>

Build the 4 channels as `[B, L]` tensors, pad the CGM and flag channels with zeros for the H future steps, and `torch.stack(..., -1)`.

</details>

In [ ]:
def tokens(w, LH, g_mu=140.0, g_sd=50.0):
    # TODO
    raise NotImplementedError

def check(f):
    w = subset(te, torch.arange(len(te["y"])) < 32)
    close(f(w, LH), neural.tokens(w), what="tokens")

tokens = use(tokens, sol.tokens, check)

## 2. GRU: a learned nonlinear recursive filter

A GRU updates a hidden state $h_t = (1-z_t)\odot h_{t-1} + z_t \odot \tilde h_t$, where the
**gate** $z_t \in (0,1)$ is computed from the input. When $z_t$ is small the state is held (a
long memory), and when it's large the state is overwritten. It's a first-order recursive filter
whose coefficient is chosen per sample by the data. Here's a minute of training from scratch on
a subset:

In [ ]:
gru = neural.build("gru")
small_tr = subset(S["train"], torch.arange(len(S["train"]["y"])) % 4 == 0)
small_va = subset(S["val"], torch.arange(len(S["val"]["y"])) % 4 == 0)
gru, hist = fit(gru, small_tr, small_va, epochs=10, lr=2e-3, bs=256, gen=torch.Generator().manual_seed(0),
                max_minutes=1.0, verbose=False)
print(f"{neural.n_params(gru)} parameters; val RMSE by epoch:", [round(h['val_rmse'], 1) for h in hist])

## 3. Attention: data-dependent weighting

Each position $i$ builds a query $q_i$; every position $j$ offers a key $k_j$ and a value $v_j$.
The output is a weighted average of the values, with weights from the query–key similarity:

$$\operatorname{Attn}(Q,K,V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V$$

The $1/\sqrt d$ keeps the dot products at unit scale, so the softmax doesn't saturate.

## Exercise 2 — `attention(q, k, v)`

Inputs are `[..., L, d]`. The softmax runs over the **key** dimension (the last one of the score
matrix).


<details><summary><b>Hint</b></summary>

`q @ k.transpose(-2, -1)` gives `[..., L, L]` scores.

</details>

In [ ]:
def attention(q, k, v):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    q, k, v = (torch.randn(2, 3, 10, 8, generator=g) for _ in range(3))
    close(f(q, k, v), F.scaled_dot_product_attention(q, k, v), tol=1e-5, what="attention")

attention = use(attention, sol.attention, check)

Our Transformer is **bidirectional**: past positions can attend to future positions. That's
allowed only because future tokens contain nothing but the planned inputs. The future CGM is
zeroed and flagged. If CGM leaked into the future tokens, attention would happily copy it.

## 4. Selective state-space models (Mamba-style)

Start from a continuous **linear** state-space model with a diagonal, stable $A$, discretised
with step $\Delta$:

$$h_t = e^{\Delta A} h_{t-1} + \Delta B\, x_t, \qquad y_t = C h_t + D x_t$$

With fixed $\Delta, B, C$ this is an **LTI filter**, so the output is a convolution of $x$ with the
kernel $K_j = C (e^{\Delta A})^j \Delta B$. Mamba's change is to make $\Delta_t, B_t, C_t$
**functions of the input**. The filter's time constant and gains then change at every step: a
large $\Delta_t$ means "reset and take in this input", and a small $\Delta_t$ means "ignore it and
keep the memory".

## Exercise 3 — `selective_scan(x, delta, A, B, C, D)`

Shapes: `x, delta [b, L, d]`, `A [d, n]` (negative), `B, C [b, L, n]`, `D [d]`. Each of the `d`
channels has its own `n`-dimensional diagonal state. Return `y [b, L, d]`.


<details><summary><b>Hint</b></summary>

Keep `h` of shape `[b, d, n]`. At step t:
`dA = exp(delta[:, t, :, None] * A)`, `dB = delta[:, t, :, None] * B[:, t, None, :]`,
`h = dA * h + dB * x[:, t, :, None]`, `y_t = (h * C[:, t, None, :]).sum(-1)`. Add `x * D` at the end.

</details>

In [ ]:
def selective_scan(x, delta, A, B, C, D):
    # TODO
    raise NotImplementedError

def check(f):
    g = torch.Generator().manual_seed(0)
    b, L, d, n_ = 2, 20, 3, 4
    x = torch.randn(b, L, d, generator=g, dtype=torch.float64)
    A = -torch.rand(d, n_, generator=g, dtype=torch.float64) - 0.1
    D = torch.randn(d, generator=g, dtype=torch.float64)
    # (a) constant delta, B, C  ->  must equal convolution with the LTI kernel
    dl, Bv, Cv = 0.5, torch.randn(n_, generator=g, dtype=torch.float64), torch.randn(n_, generator=g, dtype=torch.float64)
    K = torch.stack([(Cv * torch.exp(dl * A * j) * dl * Bv).sum(-1) for j in range(L)])   # [L, d]
    conv = torch.stack([sum(K[j] * x[:, t - j] for j in range(t + 1)) for t in range(L)], 1) + x * D
    got = f(x, torch.full((b, L, d), dl, dtype=torch.float64), A, Bv.expand(b, L, n_), Cv.expand(b, L, n_), D)
    close(got, conv, tol=1e-8, what="LTI case vs convolution")
    # (b) time-varying parameters
    dl = torch.rand(b, L, d, generator=g, dtype=torch.float64)
    Bt, Ct = torch.randn(b, L, n_, generator=g, dtype=torch.float64), torch.randn(b, L, n_, generator=g, dtype=torch.float64)
    close(f(x, dl, A, Bt, Ct, D), sol.selective_scan(x, dl, A, Bt, Ct, D), tol=1e-8, what="selective case")

selective_scan = use(selective_scan, sol.selective_scan, check)

Check (a) is the SP view in one line: **a state-space model with frozen parameters is a
convolution.** S4-style models exploit this and compute it with FFTs. Mamba can't, because its
parameters change at every step. Instead it uses a parallel *scan* in a fused GPU kernel. Our
version is a Python loop, so the cost is easy to see:

In [ ]:
b = {k: v[:256] for k, v in S["train"].items()}
for kind in ["gru", "transformer", "ssm"]:
    m = neural.build(kind)
    t = time.time(); loss = (m(b) - b["y"]).pow(2).mean(); tf = time.time() - t
    t = time.time(); loss.backward(); tb = time.time() - t
    print(f"{kind:12s} {neural.n_params(m):7d} params | forward {1000 * tf:6.0f} ms | backward {1000 * tb:6.0f} ms (batch 256, CPU)")

## 5. How they compare in the project

In [ ]:
rows = [("phase1", "gru"), ("phase3", "transformer"), ("phase3", "ssm"), ("phase1", "hybrid")]
for ph, nm in rows:
    r = json.loads(open(f"results/{ph}/{nm}.json").read())
    print(f"{nm:12s} RMSE 60 min {r['metrics']['rmse_60min']:5.1f} | epochs trained {len(r['config'].get('hist', []))}")

All were given about 6 CPU minutes. The SSM fitted in **one epoch**, so its number reflects the
budget, not the architecture. On a GPU with the real fused kernel, the ranking could be
different. That's a claim you'd have to test before making it.

## Quiz

Answer in your own words before opening the answers.

**1. Why does each token need an 'observed' flag in addition to zeroing future CGM?**

<details><summary>Answer</summary>

Zero after normalisation means 'glucose = 140 mg/dL', which is a real value. Without the flag, the model can't tell 'unknown' from '140'. The flag makes missingness explicit. The same idea handles real sensor dropouts.

</details>

**2. When is a bidirectional Transformer allowed in forecasting, and when would it be leakage?**

<details><summary>Answer</summary>

Allowed when the future positions only carry information that's legitimately known at forecast time (planned inputs, time of day). It's leakage if any future token encodes the target, e.g. unmasked future CGM, or features computed over the whole window.

</details>

**3. A linear SSM with constant Δ, B, C is a convolution. What's the kernel, and why does that matter computationally?**

<details><summary>Answer</summary>

K_j = C (e^{ΔA})^j ΔB. Being a convolution means the whole sequence can be processed in parallel with FFTs (the S4 idea) instead of a step-by-step recursion. Selective SSMs lose this because the kernel changes with time, which is why Mamba needs a parallel scan.

</details>

**4. In signal-processing terms, what does an input-dependent Δ_t do?**

<details><summary>Answer</summary>

It sets the filter's time constant per sample: the pole is e^{Δ_t A}. A large Δ_t moves the pole toward 0 (fast forgetting, strong response to the current input). A small Δ_t keeps the pole near 1 (long memory, input mostly ignored). It's an adaptive, data-controlled IIR filter bank.

</details>

**5. The SSM scored worst. Name two reasons that isn't evidence SSMs are bad for this task.**

<details><summary>Answer</summary>

1) It got one epoch within the time budget, while the others got 5–14. 2) The implementation is a slow Python loop on a CPU, not the fused kernel. Also: one seed, no hyperparameter search, and a small model. A fair comparison needs equal compute and several seeds.

</details>